In [ ]:
#include <vector>
#include <set>
#include <iostream>

In [ ]:
class taskbag{
public:
    void resize(unsigned size){
        N.resize(size); NxN.resize(size);
        unprocessed.clear();
    }
    void add(unsigned id,int n){
        N[id]=n; unprocessed.insert(id);
    }
    bool ready_to_get(){
        return unprocessed.size()==N.size();
    }
    bool get(unsigned& id, int& n){
        if(unprocessed.empty())return false;
        id = get_rand_unprocessed();  
        n = N[id];
        return true;
    }
    void put(unsigned id,int nxn){
        unprocessed.erase(id);
        NxN[id]=nxn;
    }
public:
    std::vector<int> N;
    std::vector<int> NxN;
private:
    std::set<unsigned> unprocessed;
    unsigned get_rand_unprocessed(){
        int selected = rand() % unprocessed.size();
		auto it = unprocessed.begin(); 
        for (int i = 0; i != selected; i++, it++) {}
        return *it;
    }
};

In [ ]:
{
    const int SIZE = 10;
    taskbag tbag;
    
    tbag.resize(SIZE);
    for(int i=0;i<SIZE;i++) tbag.add(i,i);
    
    while(!tbag.ready_to_get())/*wait*/;
    
    unsigned id; int N, NxN;
    while(tbag.get(id,N)){
        NxN = N*N; tbag.put(id,NxN);
    }
    
    for(int i=0;i<SIZE;i++)
        std::cout << tbag.N[i] <<"^2 = " << tbag.NxN[i] << std::endl;
}

In [ ]:
#pragma cling add_include_path("../../lib/")
#include <walimpl.hpp>
#include <syncmem.hpp>

In [ ]:
class taskbag:public templet::state{
public:
    taskbag(templet::write_ahead_log&l):state(l) { init(); }
    void resize(unsigned size){
        update(_resize, [&](std::ostream&out) {
            out << size;
		},
		[this](std::istream&in) {
            unsigned size; in >> size;
            N.resize(size); NxN.resize(size);
            unprocessed.clear();
		});   
    }
    void add(unsigned id,int n){
        update(_add, [&](std::ostream&out) {
            out << id << " " << n;
		},
		[this](std::istream&in) {
            unsigned id; int n; in >> id >> n;
            N[id]=n; unprocessed.insert(id);
		});     
    }
    bool ready_to_get(){
        update();
        return unprocessed.size()==N.size();
    }
    bool get(unsigned& id, int& n){
        bool ret;
        update(_get, [this]() { 
            if(unprocessed.empty()) _ret=false;
            else{
                _id = get_rand_unprocessed(); 
                _n = N[_id]; _ret=true; 
            }
		});
        if(_ret)n = _n; id = _id;
        return _ret;
    }
    void put(unsigned id,int nxn){
        update(_put, [&](std::ostream&out) {
            out << id << " " << nxn;
		},
		[this](std::istream&in) {
            unsigned id; int nxn; in >> id >> nxn;
            unprocessed.erase(id);
            NxN[id]=nxn;
		});  
    }
public:
    std::vector<int> N;
    std::vector<int> NxN;
private:
    std::set<unsigned> unprocessed;
    unsigned get_rand_unprocessed(){
        int selected = rand() % unprocessed.size();
		auto it = unprocessed.begin(); 
        for (int i = 0; i != selected; i++, it++) {}
        return *it;
    }
private:
	enum {_resize,_add,_get,_put};
	void on_init() override {
		resize(0);
		add(0,0);
        {unsigned id; int n; get(id,n);}
        put(0,0);
	}
    // get returns:
    bool _ret; unsigned _id; int _n; 
};

In [ ]:
{
    const int SIZE = 10;

    templet::write_ahead_log wal;
    taskbag tbag(wal);
    
    tbag.resize(SIZE);
    for(int i=0;i<SIZE;i++) tbag.add(i,i);
    
    while(!tbag.ready_to_get())/*wait*/;

    unsigned id; int N, NxN; 
    while(tbag.get(id,N)){
        NxN = N*N; tbag.put(id,NxN);
    }
    
    for(int i=0;i<SIZE;i++)
        std::cout << tbag.N[i] <<"^2 = " << tbag.NxN[i] << std::endl;
}